# LCEL Exploration

LangChain Expression Language (LCEL) uses the `|` operator to chain *Runnables*.
A Runnable is anything with an `.invoke(input)` method.
The pipe connects them so the output of one becomes the input of the next.

This notebook builds up the RAG chain pattern from scratch using toy examples
before writing the real pipeline.

**Contents**
1. `RunnableLambda` — wrap any Python function
2. `RunnablePassthrough` — identity, passes input unchanged
3. Piping runnables with `|`
4. `RunnableParallel` — fan out one input to many functions, merge results
5. The standard RAG pattern: `{context, question} | prompt | llm | parser`
6. Structured output — carrying source chunks through the chain


In [1]:
import sys
sys.path.insert(0, '..')

from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

print('imports ok')


/Users/harsh/drive/workspace/ai_projects/doc-rag-engine/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports ok


---
## 1. RunnableLambda

`RunnableLambda` wraps any Python callable into a Runnable so it can participate
in a chain. The function receives whatever the previous step output.


In [2]:
shout   = RunnableLambda(str.upper)
reverse = RunnableLambda(lambda s: s[::-1])
words   = RunnableLambda(str.split)

print(shout.invoke('hello world'))    # HELLO WORLD
print(reverse.invoke('hello'))        # olleh
print(words.invoke('one two three'))  # ['one', 'two', 'three']


HELLO WORLD
olleh
['one', 'two', 'three']


In [3]:
# A function with logic is fine too
def word_count(text: str) -> dict:
    tokens = text.lower().split()
    return {'text': text, 'count': len(tokens), 'unique': len(set(tokens))}

counter = RunnableLambda(word_count)
print(counter.invoke('the cat sat on the mat'))
# {'text': 'the cat sat on the mat', 'count': 6, 'unique': 5}


{'text': 'the cat sat on the mat', 'count': 6, 'unique': 5}


---
## 2. RunnablePassthrough

`RunnablePassthrough` returns its input unchanged. It sounds useless — but
inside `RunnableParallel` it is how you carry the original question forward
while other branches do work on it.


In [4]:
passthrough = RunnablePassthrough()

print(passthrough.invoke('the original question'))  # the original question
print(passthrough.invoke({'key': 'value'}))         # {'key': 'value'}
# Input type does not matter. Whatever comes in comes out.


the original question
{'key': 'value'}


---
## 3. Piping runnables with `|`

The `|` operator creates a chain where the output of the left side becomes
the input of the right side. The chain itself is also a Runnable.


In [5]:
# Each step transforms the value
pipeline = (
    RunnableLambda(str.strip)
    | RunnableLambda(str.lower)
    | RunnableLambda(str.split)
    | RunnableLambda(len)
)

print(pipeline.invoke('  Hello World  '))   # 2
print(pipeline.invoke('  One Two Three  ')) # 3

# The chain is itself a Runnable with invoke/stream/batch.
print(type(pipeline))


2
3
<class 'langchain_core.runnables.base.RunnableSequence'>


In [6]:
# Steps can return any type — the next step just has to accept that type.
# String -> list -> dict -> string
summarise = (
    RunnableLambda(str.split)
    | RunnableLambda(lambda tokens: {'words': tokens, 'n': len(tokens)})
    | RunnableLambda(lambda d: f"{d['n']} words: {', '.join(d['words'][:3])}...")
)

print(summarise.invoke('the quick brown fox jumps over the lazy dog'))
# 9 words: the, quick, brown...


9 words: the, quick, brown...


---
## 4. RunnableParallel

`RunnableParallel` is the key piece for RAG. It takes **one input** and fans it
out to multiple runnables simultaneously. Each branch runs independently and
the results are merged into a dict.

```
         input
           |
     ------+------
     |            |
  branch_a     branch_b
     |            |
     +------+------
            |
   {"a": result_a, "b": result_b}
```


In [7]:
parallel = RunnableParallel({
    'upper':   RunnableLambda(str.upper),
    'length':  RunnableLambda(len),
    'words':   RunnableLambda(str.split),
    'original': RunnablePassthrough(),
})

result = parallel.invoke('hello world')
print(result)
# {'upper': 'HELLO WORLD', 'length': 11, 'words': ['hello', 'world'], 'original': 'hello world'}


{'upper': 'HELLO WORLD', 'length': 11, 'words': ['hello', 'world'], 'original': 'hello world'}


In [8]:
# The dict shorthand is equivalent -- LangChain wraps plain functions automatically.
parallel2 = RunnableParallel({
    'upper':    str.upper,        # plain function
    'length':   len,              # built-in
    'original': RunnablePassthrough(),
})

print(parallel2.invoke('lcel is elegant'))


{'upper': 'LCEL IS ELEGANT', 'length': 15, 'original': 'lcel is elegant'}


In [9]:
# RunnableParallel can be used inline with {} syntax inside a chain.
# LCEL interprets a dict as RunnableParallel automatically.
chain = (
    {'upper': str.upper, 'length': len}
    | RunnableLambda(lambda d: f"'{d['upper']}' has {d['length']} chars")
)

print(chain.invoke('transformer'))  # 'TRANSFORMER' has 11 chars


'TRANSFORMER' has 11 chars


---
## 5. The standard LCEL RAG pattern

Now apply this to the actual RAG problem. The chain needs two things before
it can format the prompt: the retrieved context and the original question.

```
question (str)
        |
        +-- RunnableLambda(retrieve)  --> context string
        +-- RunnablePassthrough()     --> question string (unchanged)
        |
        v
{"context": "...", "question": "..."}
        |
        v
ChatPromptTemplate  (slots context + question into the template)
        |
        v
LLM
        |
        v
StrOutputParser
```

We mock the LLM here so this cell runs without Ollama.


In [10]:
# Mock retriever: returns a formatted context string
FAKE_CORPUS = {
    'attention': '[1] Source: attention_need.pdf\nMulti-head attention allows the model to jointly '
                 'attend to information from different representation subspaces.',
    'ragas':     '[1] Source: ragas.pdf\nFaithfulness measures how factually consistent the '
                 'generated answer is with the retrieved context.',
    'default':   '[1] Source: rag_llm.pdf\nRetrieval-augmented generation reduces hallucination '
                 'by grounding answers in retrieved documents.',
}

def fake_retrieve(question: str) -> str:
    q = question.lower()
    if 'attention' in q:
        return FAKE_CORPUS['attention']
    if 'ragas' in q or 'faithfulness' in q:
        return FAKE_CORPUS['ragas']
    return FAKE_CORPUS['default']

print(fake_retrieve('How does multi-head attention work?'))


[1] Source: attention_need.pdf
Multi-head attention allows the model to jointly attend to information from different representation subspaces.


In [11]:
# Build the prompt template
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer using only the context. Cite sources by number.'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}'),
])

# Show what the prompt produces before any LLM call
formatted = prompt.invoke({
    'context': fake_retrieve('How does multi-head attention work?'),
    'question': 'How does multi-head attention work?',
})

for msg in formatted.messages:
    print(f'[{msg.type}]')
    print(msg.content)
    print()


[system]
Answer using only the context. Cite sources by number.

[human]
Context:
[1] Source: attention_need.pdf
Multi-head attention allows the model to jointly attend to information from different representation subspaces.

Question: How does multi-head attention work?



In [12]:
# Mock LLM: returns a canned response as an AIMessage
from langchain_core.messages import AIMessage

def fake_llm(prompt_value) -> AIMessage:
    # In a real chain the prompt_value is a ChatPromptValue.
    # We extract the human message and pretend the LLM answered it.
    human_msg = prompt_value.messages[-1].content
    question = human_msg.split('Question:')[-1].strip()
    return AIMessage(content=f'[mock answer to: "{question}"] Based on the context [1], ...')

# Wire up the full chain
rag_chain = (
    {
        'context':  RunnableLambda(fake_retrieve),
        'question': RunnablePassthrough(),
    }
    | prompt
    | RunnableLambda(fake_llm)
    | StrOutputParser()
)

answer = rag_chain.invoke('How does multi-head attention work?')
print(answer)


[mock answer to: "How does multi-head attention work?"] Based on the context [1], ...


In [13]:
# Try a different question to confirm routing works
print(rag_chain.invoke('What does RAGAS measure?'))
print()
print(rag_chain.invoke('What is RAG?'))


[mock answer to: "What does RAGAS measure?"] Based on the context [1], ...

[mock answer to: "What is RAG?"] Based on the context [1], ...


---
## 6. Structured output — carrying source chunks through the chain

The real pipeline returns more than a string. The final output is:

```python
{
    'answer':           str,              # LLM response
    'source_chunks':    list[dict],       # the retrieved chunks with metadata
    'retrieval_method': str,              # 'reranked'
}
```

The challenge: by the time the LLM generates its answer, the source chunks
have already been consumed and formatted into a string. We need to carry them
forward alongside the LLM call.

`RunnablePassthrough.assign` is the LCEL idiom for this: it adds a key to the
running state dict without discarding what is already there.


In [14]:
# Demonstrate RunnablePassthrough.assign
# assign adds keys to the state dict while passing everything else through.

step1 = RunnablePassthrough.assign(
    upper=RunnableLambda(str.upper),
    length=RunnableLambda(len),
)

# Input is a plain string; .assign wraps it as {'input': str} first, then adds keys.
# But assign needs a dict input. So we first convert the string to a dict:
to_dict  = RunnableLambda(lambda q: {'question': q})
with_extras = to_dict | RunnablePassthrough.assign(
    upper=lambda d: d['question'].upper(),
    length=lambda d: len(d['question']),
)

print(with_extras.invoke('hello world'))
# {'question': 'hello world', 'upper': 'HELLO WORLD', 'length': 11}


{'question': 'hello world', 'upper': 'HELLO WORLD', 'length': 11}


In [15]:
# A retriever that returns BOTH the formatted string AND the raw chunk objects.
# This is the real pattern: retrieve once, return both representations.

from dataclasses import dataclass

@dataclass
class FakeChunk:
    text: str
    filename: str
    score: float
    retrieval_method: str = 'reranked'

FAKE_CHUNKS = [
    FakeChunk('Multi-head attention allows joint attention.', 'attention_need.pdf', 0.997),
    FakeChunk('The transformer stacks N encoder layers.', 'attention_need.pdf', 0.982),
]

def retrieve_with_chunks(question: str) -> dict:
    """Returns both the formatted context string and the raw chunk objects."""
    chunks = FAKE_CHUNKS  # in reality: hybrid.retrieve_rrf -> reranker.rerank
    context_str = '\n\n'.join(
        f'[{i}] Source: {c.filename}\n{c.text}'
        for i, c in enumerate(chunks, 1)
    )
    return {'context_str': context_str, 'chunks': chunks}

print(retrieve_with_chunks('How does attention work?'))


{'context_str': '[1] Source: attention_need.pdf\nMulti-head attention allows joint attention.\n\n[2] Source: attention_need.pdf\nThe transformer stacks N encoder layers.', 'chunks': [FakeChunk(text='Multi-head attention allows joint attention.', filename='attention_need.pdf', score=0.997, retrieval_method='reranked'), FakeChunk(text='The transformer stacks N encoder layers.', filename='attention_need.pdf', score=0.982, retrieval_method='reranked')]}


In [16]:
# Full structured-output chain using RunnablePassthrough.assign to thread
# chunks alongside the LLM call.

def build_structured_prompt(state: dict) -> dict:
    """Format the ChatPromptValue and carry chunks forward."""
    pv = prompt.invoke({
        'context':  state['retrieval']['context_str'],
        'question': state['question'],
    })
    return {'prompt_value': pv, 'chunks': state['retrieval']['chunks']}

def call_llm(state: dict) -> dict:
    """Call the LLM and carry chunks forward."""
    ai_msg = fake_llm(state['prompt_value'])  # replace with ChatOllama in real pipeline
    return {'answer': ai_msg.content, 'chunks': state['chunks']}

def structure_output(state: dict) -> dict:
    """Produce the final structured result."""
    return {
        'answer': state['answer'],
        'source_chunks': [
            {'text': c.text, 'filename': c.filename, 'score': c.score}
            for c in state['chunks']
        ],
        'retrieval_method': state['chunks'][0].retrieval_method if state['chunks'] else 'unknown',
    }

structured_chain = (
    RunnableLambda(lambda q: {'question': q, 'retrieval': retrieve_with_chunks(q)})
    | RunnableLambda(build_structured_prompt)
    | RunnableLambda(call_llm)
    | RunnableLambda(structure_output)
)

result = structured_chain.invoke('How does multi-head attention work?')
import pprint
pprint.pprint(result)


{'answer': '[mock answer to: "How does multi-head attention work?"] Based on '
           'the context [1], ...',
 'retrieval_method': 'reranked',
 'source_chunks': [{'filename': 'attention_need.pdf',
                    'score': 0.997,
                    'text': 'Multi-head attention allows joint attention.'},
                   {'filename': 'attention_need.pdf',
                    'score': 0.982,
                    'text': 'The transformer stacks N encoder layers.'}]}


---
## 7. What the real pipeline.py will look like

Now that each piece is understood, here is the chain structure that
`src/generation/pipeline.py` will use — same pattern as above, but with
the real components plugged in.

```python
self._chain = (
    # Step 1: embed question, run hybrid retrieval, rerank
    # Returns {'question': str, 'retrieval': {'context_str': str, 'chunks': list}}
    RunnableLambda(self._retrieve)

    # Step 2: format ChatPromptValue, carry chunks forward
    | RunnableLambda(self._build_prompt_state)

    # Step 3: call ChatOllama, carry chunks forward
    | RunnableLambda(self._call_llm)

    # Step 4: assemble final structured output
    | RunnableLambda(self._structure_output)
)
```

Where each step method returns a dict that the next step reads from.
Chunks are threaded through as a key in that dict — never lost, never
re-fetched.

The output of `chain.invoke(question)` is:
```python
{
    'answer':           'Multi-head attention allows... [1]',
    'source_chunks':    [{'text': '...', 'filename': 'attention_need.pdf', 'score': 0.997}, ...],
    'retrieval_method': 'reranked',
}
```


In [17]:
# Verify streaming works on an LCEL chain (important for the real pipeline).
# With a real LLM the stream() call yields tokens; with our mock it yields one chunk.

simple_chain = (
    {'context': RunnableLambda(fake_retrieve), 'question': RunnablePassthrough()}
    | prompt
    | RunnableLambda(fake_llm)
    | StrOutputParser()
)

print('Streaming tokens:')
for token in simple_chain.stream('What is RAG?'):
    print(repr(token), end=' ')
print()

# With ChatOllama the stream() call yields actual tokens one by one.
# Usage pattern in the real pipeline:
#
#   for token in pipeline.stream('What is RAG?'):
#       print(token, end='', flush=True)
#   print()


Streaming tokens:
'[mock answer to: "What is RAG?"] Based on the context [1], ...' 
